# 04 — Ageing Axis and Perturbation Validation

Constructs a continuous ageing score from a data-derived gene signature, builds
the ageing gradient, simulates all fifteen Yamanaka-factor combinations, and
validates the predicted transcriptional effects against those measured in the
screen, using gene-shuffled and combination-mismatched null models.

In [ ]:
import sys
!{sys.executable} -m pip install --no-deps git+https://github.com/morris-lab/CellOracle.git
!{sys.executable} -m pip install "numpy==1.26.4" anndata scanpy genomepy gimmemotifs goatools igraph jupyter louvain pybedtools velocyto
!{sys.executable} -m pip install fa2-modified

from google.colab import drive
drive.mount('/content/drive')

  Cloning https://github.com/morris-lab/CellOracle.git to /tmp/pip-req-build-ww115o6e
  Running command git clone --filter=blob:none --quiet https://github.com/morris-lab/CellOracle.git /tmp/pip-req-build-ww115o6e
  Resolved https://github.com/morris-lab/CellOracle.git to commit 7948870a3b70f7d228e54734e1fa9ed3291fc23b
  Preparing metadata (setup.py) ... done
  Created wheel for celloracle: filename=celloracle-0.22.0-py3-none-any.whl size=12372847 sha256=6b722bc07771e00cb8fde5d673bac6046e6c3d4ba8b03686aec0b848ef4bb312
  Stored in directory: /tmp/pip-ephem-wheel-cache-hyv4gknb/wheels/cc/aa/03/95b3761e1aa553e81e2506c7f64cf5878e46cad7f5eeffd09d
Successfully built celloracle
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 78.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 45

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np

adata = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")
print(adata.shape)
print(adata.obs['age'].value_counts())

# restore raw counts and normalise (the stored X is scaled; scoring requires
# normalised, log-transformed values)
adata.X = adata.layers['raw_count'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("X min:", adata.X.min(), "| max:", adata.X.max())

(9977, 2000)
age
Young    7544
Aged     2433
Name: count, dtype: int64
X min: 0.0 | max: 9.02534


## Deriving the ageing signature

`rank_genes_groups` defaults to `adata.raw`; `use_raw=False` is set so the
comparison runs on the processed matrix rather than the stored raw object.

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby='age', groups=['Aged'], reference='Young',
    method='wilcoxon', use_raw=False
)

# exclude ribosomal, mitochondrial, heat-shock and predicted genes, which
# reflect technical rather than biological variation
ranked = list(adata.uns['rank_genes_groups']['names']['Aged'])
aged_signature_datadriven = [
    g for g in ranked
    if not g.startswith(('Rps', 'Rpl', 'mt-', 'Hsp', 'Gm')) and g != 'Malat1'
][:50]

print(f"data-driven signature: {len(aged_signature_datadriven)} genes")
print(aged_signature_datadriven[:20])

data-driven signature: 50 genes
['Ass1', 'Dstn', 'Anxa3', 'Actb', 'Plp2', 'S100a4', 'Ctla2a', 'Pola2', 'Tpm2', 'Fhl3', 'Acta2', 'Tagln', 'Map1b', 'Taldo1', 'Pdpn', 'Esd', 'Cnn1', '2200002D01Rik', 'Gsto1', 'Inhba']


## Comparison with a published senescence panel

The data-derived signature is compared against a published SASP/senescence panel
to test whether classical senescence markers track ageing in these cells.

In [ ]:
# data-driven score
sc.tl.score_genes(adata, gene_list=aged_signature_datadriven,
                  score_name='aging_score_datadriven', use_raw=False)

# published senescence/SASP panel, scored via .raw so the full panel is available
published_full = ['Cdkn1a','Cdkn2a','Trp53','Serpine1','Glb1','Il6','Il1b','Il1a','Tnf',
                  'Cxcl1','Cxcl2','Ccl2','Ccl8','Cxcl10','Mmp3','Mmp12','Mmp13','Timp1',
                  'Igfbp3','Igfbp4','B2m','Nfkb1','Gdf15']
present_in_raw = [g for g in published_full if g in adata.raw.var_names]
print(f"published genes available: {len(present_in_raw)}/23")
sc.tl.score_genes(adata, gene_list=present_in_raw,
                  score_name='aging_score_published', use_raw=True)

print(adata.obs[['aging_score_datadriven','aging_score_published']].describe())

published genes available: 20/23
       aging_score_datadriven  aging_score_published
count             9977.000000            9977.000000
mean                 0.758478               0.403571
std                  0.542877               0.184779
min                 -0.327471              -0.144212
25%                  0.326230               0.270153
50%                  0.683939               0.386641
75%                  1.121088               0.517440
max                  2.550024               1.465634


In [ ]:
from scipy.stats import pearsonr
r, p = pearsonr(adata.obs['aging_score_datadriven'], adata.obs['aging_score_published'])
print(f"correlation between the two ageing scores: r={r:.3f}, p={p:.2e}")

for score in ['aging_score_datadriven', 'aging_score_published']:
    young = adata.obs.loc[adata.obs['age']=='Young', score]
    aged  = adata.obs.loc[adata.obs['age']=='Aged',  score]
    print(f"{score}: Young={young.mean():.3f}, Aged={aged.mean():.3f}, diff={aged.mean()-young.mean():+.3f}")

correlation between the two ageing scores: r=-0.321, p=1.19e-238
aging_score_datadriven: Young=0.630, Aged=1.157, diff=+0.527
aging_score_published: Young=0.420, Aged=0.352, diff=-0.068


In [ ]:
import numpy as np
pub_present = ['Cdkn1a','Cdkn2a','Serpine1','Il6','Cxcl1','Cxcl2','Ccl2','Ccl8',
               'Cxcl10','Mmp3','Mmp13','Timp1','Igfbp3','Igfbp4','Gdf15']
expr = adata.raw[:, [g for g in pub_present if g in adata.raw.var_names]].X
print("mean expression of published genes:", np.asarray(expr.mean(axis=0)).flatten().round(2))

# the senescence core specifically, by age
for g in ['Cdkn1a','Cdkn2a','Serpine1']:
    if g in adata.raw.var_names:
        idx = list(adata.raw.var_names).index(g)
        young = np.asarray(adata.raw[adata.obs['age']=='Young'].X[:, idx].mean()).item()
        aged  = np.asarray(adata.raw[adata.obs['age']=='Aged'].X[:, idx].mean()).item()
        print(f"{g}: Young={young:.3f}, Aged={aged:.3f}")

mean expression of published genes: [1.52 1.13 1.02 0.11 0.29 0.   1.21 0.82 0.07 0.32 0.12 3.07 0.12 2.46
 0.12]
Cdkn1a: Young=1.550, Aged=1.448
Cdkn2a: Young=1.074, Aged=1.301
Serpine1: Young=0.971, Aged=1.169


In [ ]:
import pandas as pd
comp = pd.crosstab(adata.obs['cell_type'], adata.obs['age'], normalize='columns')
print((comp*100).round(1))

age                     Young  Aged
cell_type                          
Adipogenic                0.6   0.0
Fibroblast/stromal       54.9  40.8
Inflammatory/secretory    0.0   2.8
Interferon                1.9   0.8
Myofibroblast             8.7  22.8
Oxidative-stress          0.9   2.6
Proliferating             6.7   9.9
Senescent/stressed        6.3  11.5
Tendon                   19.9   8.8


In [ ]:
import anndata

print(adata.obs['aging_score_datadriven'].describe())
print("NaNs:", adata.obs['aging_score_datadriven'].isna().sum())

# allow writing nullable string columns (present in adata.obs)
anndata.settings.allow_write_nullable_strings = True

adata.write_h5ad("/content/drive/MyDrive/roux_project/msc_aging_scored.h5ad")
print("saved")

count    9977.000000
mean        0.758478
std         0.542877
min        -0.327471
25%         0.326230
50%         0.683939
75%         1.121088
max         2.550024
Name: aging_score_datadriven, dtype: float64
NaNs: 0
saved


## Ageing gradient and perturbation simulation

The ageing score is supplied to CellOracle's gradient framework in place of
pseudotime, yielding a vector field describing the direction of increasing
transcriptional age. All fifteen factor combinations are then simulated.

In [ ]:
import celloracle as co

oracle = co.load_hdf5("/content/drive/MyDrive/roux_project/oracle_roux_v2.celloracle.oracle")
links = co.load_hdf5("/content/drive/MyDrive/roux_project/links_roux_v2.celloracle.links")

print("order matches?", (oracle.adata.obs_names == adata.obs_names).all())
oracle.adata.obs['aging_score'] = adata.obs['aging_score_datadriven'].values
print("ageing score attached, mean:", oracle.adata.obs['aging_score'].mean().round(3))

oracle.to_hdf5("/content/drive/MyDrive/roux_project/oracle_roux_v2.celloracle.oracle")
print("oracle re-saved with ageing score")

22:31:17 - INFO - Creating new config.
22:31:17 - INFO - Using included version of AMD.
22:31:17 - INFO - Using included version of BioProspector.
22:31:17 - INFO - Using included version of ChIPMunk.
22:31:17 - WARNING - DiNAMO not found. To include it you will have to install it.
22:31:17 - WARNING - DREME not found. To include it you will have to install it.
22:31:17 - WARNING - GADEM not found. To include it you will have to install it.
22:31:17 - INFO - Using included version of HMS.
22:31:17 - WARNING - Homer not found. To include it you will have to install it.
22:31:17 - INFO - Using included version of Improbizer.
22:31:17 - INFO - Using included version of MDmodule.
22:31:17 - WARNING - MEME not found. To include it you will have to install it.
22:31:17 - WARNING - MEMEW not found. To include it you will have to install it.
22:31:17 - INFO - Using included version of MotifSampler.
22:31:17 - INFO - Using included version of Posmo.
22:31:17 - WARNING - ProSampler not found. To

order matches? True
ageing score attached, mean: 0.758
oracle re-saved with ageing score


In [ ]:
from celloracle.applications import Gradient_calculator
import numpy as np

gradient = Gradient_calculator(oracle_object=oracle, pseudotime_key="aging_score")
gradient.calculate_p_mass(smooth=0.8, n_grid=40, n_neighbors=200)
gradient.calculate_mass_filter(min_mass=0.01, plot=False)
gradient.transfer_data_into_grid(args={"method": "polynomial", "n_poly": 3}, plot=False)
gradient.calculate_gradient()
print("ageing gradient built")

# confirm the gradient is working from the ageing score values
print(f"gradient value range: {np.nanmin(gradient.pseudotime):.3f} to {np.nanmax(gradient.pseudotime):.3f}")
print(f"ageing score range:   {oracle.adata.obs['aging_score'].min():.3f} to {oracle.adata.obs['aging_score'].max():.3f}")

ageing gradient built
gradient value range: -0.327 to 2.550
ageing score range:   -0.327 to 2.550


In [ ]:
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)
print("GRN fitted for simulation")

  0%|          | 0/9 [00:00<?, ?it/s]

GRN fitted for simulation


In [ ]:
import pickle, os
import numpy as np

# overexpression values set to approximately twice each factor's observed maximum
factor_map = {'S':('Sox2',1.3), 'O':('Pou5f1',0.2), 'K':('Klf4',6.0), 'M':('Myc',3.6)}
combinations = ['S','O','K','M','SO','SK','SM','OK','OM','KM','SOK','SOM','SKM','OKM','SOKM']
cell_types = list(oracle.adata.obs['cell_type'].unique())
save_path = "/content/drive/MyDrive/roux_project/sim_results.pkl"

results = pickle.load(open(save_path,'rb')) if os.path.exists(save_path) else {}

for combo in combinations:
    if combo in results:
        print(f"{combo}: already computed")
        continue
    perturb = {factor_map[l][0]: factor_map[l][1] for l in combo}
    oracle.simulate_shift(perturb_condition=perturb, n_propagation=3, clip_delta_X=True)
    delta = np.asarray(oracle.adata.layers['delta_X'])
    # store per-cell-type averages rather than full per-cell matrices
    results[combo] = {ct: delta[(oracle.adata.obs['cell_type']==ct).values].mean(axis=0)
                      for ct in cell_types}
    pickle.dump(results, open(save_path,'wb'))
    print(f"{combo}: simulated")

print(f"\nall {len(combinations)} combinations simulated")

S: already computed
O: already computed
K: already computed
M: already computed
SO: already computed
SK: already computed
SM: already computed
OK: already computed
OM: already computed
KM: already computed
SOK: already computed
SOM: already computed
SKM: already computed
OKM: already computed
SOKM: already computed

all 15 combinations simulated


## Predicted-versus-measured validation

Predicted per-gene expression changes are compared to those measured in the
screen (treated minus untreated cells), for each combination and cell
population. Two null models test whether the correspondence exceeds chance.

In [ ]:
import numpy as np

# restore normalised, log-transformed values for the measured comparison
adata_m = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")
adata_m.X = adata_m.layers['raw_count'].copy()
sc.pp.normalize_total(adata_m, target_sum=1e4)
sc.pp.log1p(adata_m)

# align to the genes used in the simulation
sim_genes = list(oracle.adata.var_names)
adata_matched = adata_m[:, [g for g in sim_genes if g in adata_m.var_names]].copy()
matched_genes = list(adata_matched.var_names)
print(f"genes matched between measured data and simulation: {len(matched_genes)}")

combos = ['S','O','K','M','SO','SK','SM','OK','OM','KM','SOK','SOM','SKM','OKM','SOKM']
measured = {}
for combo in combos:
    per_type = {}
    for ct in adata_matched.obs['cell_type'].unique():
        nt_mask    = ((adata_matched.obs['combination_short']=='NT') & (adata_matched.obs['cell_type']==ct)).values
        treat_mask = ((adata_matched.obs['combination_short']==combo) & (adata_matched.obs['cell_type']==ct)).values
        if nt_mask.sum() >= 5 and treat_mask.sum() >= 5:
            nt_mean    = np.asarray(adata_matched.X[nt_mask].mean(axis=0)).flatten()
            treat_mean = np.asarray(adata_matched.X[treat_mask].mean(axis=0)).flatten()
            per_type[ct] = treat_mean - nt_mean
    measured[combo] = per_type
    print(f"{combo}: {len(per_type)} cell types with sufficient cells")

genes matched between measured data and simulation: 1966
S: 5 cell types with sufficient cells
O: 7 cell types with sufficient cells
K: 7 cell types with sufficient cells
M: 7 cell types with sufficient cells
SO: 7 cell types with sufficient cells
SK: 6 cell types with sufficient cells
SM: 6 cell types with sufficient cells
OK: 6 cell types with sufficient cells
OM: 6 cell types with sufficient cells
KM: 6 cell types with sufficient cells
SOK: 6 cell types with sufficient cells
SOM: 6 cell types with sufficient cells
SKM: 6 cell types with sufficient cells
OKM: 6 cell types with sufficient cells
SOKM: 6 cell types with sufficient cells


In [ ]:
from scipy.stats import pearsonr
import pandas as pd

sim_gene_idx = {g: i for i, g in enumerate(oracle.adata.var_names)}
matched_idx_in_sim = [sim_gene_idx[g] for g in matched_genes]

correlations = []
print(f"{'combo':<6}{'cell_type':<24}{'r':>8}")
print("-"*38)
for combo in combos:
    for ct in measured[combo]:
        measured_vec  = measured[combo][ct]
        predicted_vec = results[combo][ct][matched_idx_in_sim]
        r, p = pearsonr(predicted_vec, measured_vec)
        correlations.append({'combo': combo, 'cell_type': ct, 'r': r, 'p': p})
        if ct in ['Fibroblast/stromal','Tendon','Myofibroblast']:
            print(f"{combo:<6}{ct:<24}{r:>8.3f}")

cor_df = pd.DataFrame(correlations)
print("\n=== SUMMARY ===")
print(f"mean r ({len(cor_df)} combination x cell-type pairs): {cor_df['r'].mean():.3f}")
print(f"median r: {cor_df['r'].median():.3f}")
print(f"% positive: {(cor_df['r']>0).mean()*100:.0f}%")
print(f"% with r>0.2: {(cor_df['r']>0.2).mean()*100:.0f}%")

combo cell_type                      r
--------------------------------------
S     Fibroblast/stromal         0.044
S     Myofibroblast              0.031
S     Tendon                     0.089
O     Fibroblast/stromal         0.068
O     Myofibroblast              0.004
O     Tendon                    -0.138
K     Fibroblast/stromal         0.103
K     Myofibroblast              0.238
K     Tendon                     0.249
M     Fibroblast/stromal        -0.178
M     Myofibroblast              0.053
M     Tendon                    -0.087
SO    Fibroblast/stromal         0.087
SO    Myofibroblast              0.011
SO    Tendon                     0.072
SK    Fibroblast/stromal         0.255
SK    Myofibroblast              0.097
SK    Tendon                     0.152
SM    Fibroblast/stromal         0.012
SM    Myofibroblast              0.150
SM    Tendon                     0.136
OK    Fibroblast/stromal         0.170
OK    Myofibroblast              0.151
OK    Tendon             

In [ ]:
from scipy.stats import mannwhitneyu

# Null 1: permute gene identities within each prediction
np.random.seed(0)
shuffled_rs = []
for combo in combos:
    for ct in measured[combo]:
        predicted_vec = results[combo][ct][matched_idx_in_sim]
        r, _ = pearsonr(np.random.permutation(predicted_vec), measured[combo][ct])
        shuffled_rs.append(r)

# Null 2: pair each combination's prediction with a different combination's measurement
mismatch_rs = []
for i, combo in enumerate(combos):
    wrong_combo = combos[(i+1) % len(combos)]
    for ct in measured[combo]:
        if ct in results[wrong_combo]:
            r, _ = pearsonr(results[wrong_combo][ct][matched_idx_in_sim], measured[combo][ct])
            mismatch_rs.append(r)

print("=== NULL MODEL COMPARISON ===")
print(f"Real (matched):              mean r = {cor_df['r'].mean():.3f}")
print(f"Null 1 (shuffled genes):     mean r = {np.mean(shuffled_rs):.3f}")
print(f"Null 2 (wrong combination):  mean r = {np.mean(mismatch_rs):.3f}")

_, p1 = mannwhitneyu(cor_df['r'], shuffled_rs, alternative='greater')
_, p2 = mannwhitneyu(cor_df['r'], mismatch_rs, alternative='greater')
print(f"\nreal > shuffled-genes:   p = {p1:.2e}")
print(f"real > wrong-combination: p = {p2:.2e}")

=== NULL MODEL COMPARISON ===
Real (matched):              mean r = 0.133
Null 1 (shuffled genes):     mean r = -0.000
Null 2 (wrong combination):  mean r = 0.128

real > shuffled-genes:   p = 1.84e-11
real > wrong-combination: p = 4.74e-01


In [ ]:
print("Similarity of MEASURED combination effects to one another:\n")
for ct in ['Fibroblast/stromal', 'Tendon', 'Myofibroblast']:
    combos_here = [c for c in combos if ct in measured[c]]
    pairwise_rs = [pearsonr(measured[combos_here[i]][ct], measured[combos_here[j]][ct])[0]
                   for i in range(len(combos_here)) for j in range(i+1, len(combos_here))]
    print(f"{ct}: mean pairwise r = {np.mean(pairwise_rs):.3f} "
          f"(range {np.min(pairwise_rs):.2f} to {np.max(pairwise_rs):.2f})")

print(f"\nFor comparison, prediction accuracy (mean r): {cor_df['r'].mean():.3f}")

Similarity of MEASURED combination effects to one another:

Fibroblast/stromal: mean pairwise r = 0.826 (range 0.72 to 0.93)
Tendon: mean pairwise r = 0.864 (range 0.68 to 0.95)
Myofibroblast: mean pairwise r = 0.504 (range 0.07 to 0.79)

For comparison, prediction accuracy (mean r): 0.133


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# A: prediction accuracy per combination (Klf4-containing highlighted)
ax = axes[0,0]
combo_means = cor_df.groupby('combo')['r'].mean().reindex(combos)
colors = ['#d62728' if 'K' in c else '#7f7f7f' for c in combo_means.index]
ax.bar(range(len(combo_means)), combo_means.values, color=colors)
ax.set_xticks(range(len(combo_means)))
ax.set_xticklabels(combo_means.index, rotation=45, ha='right')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Mean prediction accuracy (r)')
ax.set_title('A. Prediction accuracy by combination\n(red = contains Klf4)')

# B: null model comparison
ax = axes[0,1]
bar_data = [cor_df['r'].mean(), np.mean(shuffled_rs), np.mean(mismatch_rs)]
bar_labels = ['Real\n(matched)', 'Null 1\n(shuffled genes)', 'Null 2\n(wrong combination)']
bars = ax.bar(bar_labels, bar_data, color=['#2ca02c', '#c7c7c7', '#ff7f0e'])
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Mean correlation (r)')
ax.set_title('B. Null model comparison\nExceeds shuffled-genes (p=2e-11), not wrong-combination (p=0.47)')
for bar, val in zip(bars, bar_data):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}', ha='center', fontsize=10)

# C: accuracy across combinations x cell types
ax = axes[1,0]
big_types = ['Fibroblast/stromal','Tendon','Myofibroblast','Proliferating','Senescent/stressed']
heat = np.full((len(combos), len(big_types)), np.nan)
for i, combo in enumerate(combos):
    for j, ct in enumerate(big_types):
        match = cor_df[(cor_df['combo']==combo)&(cor_df['cell_type']==ct)]
        if len(match): heat[i,j] = match['r'].values[0]
im = ax.imshow(heat, cmap='RdBu_r', vmin=-0.35, vmax=0.35, aspect='auto')
ax.set_xticks(range(len(big_types))); ax.set_xticklabels(big_types, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(combos))); ax.set_yticklabels(combos, fontsize=8)
ax.set_title('C. Prediction accuracy (r)\nacross combinations and cell types')
plt.colorbar(im, ax=ax, label='r', fraction=0.046)

# D: measured combinations resemble one another
ax = axes[1,1]
sim_by_type = {}
for ct in ['Fibroblast/stromal','Tendon','Myofibroblast']:
    combos_here = [c for c in combos if ct in measured[c]]
    sim_by_type[ct] = [pearsonr(measured[combos_here[i]][ct], measured[combos_here[j]][ct])[0]
                       for i in range(len(combos_here)) for j in range(i+1, len(combos_here))]
ax.boxplot([sim_by_type[ct] for ct in sim_by_type], tick_labels=[ct.split('/')[0] for ct in sim_by_type])
ax.axhline(cor_df['r'].mean(), color='#2ca02c', ls='--', label=f'Prediction accuracy ({cor_df["r"].mean():.2f})')
ax.set_ylabel('Correlation between\ndifferent combinations (measured)')
ax.set_title('D. Measured combinations resemble one another\n(r~0.83): a shared reprogramming programme')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/roux_project/predicted_vs_measured_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print("figure saved")

figure saved
